In [19]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [20]:
columns = [
    "id",
    "diagnosis",

    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "smoothness_mean",
    "compactness_mean",
    "concavity_mean",
    "concave_points_mean",
    "symmetry_mean",
    "fractal_dimension_mean",

    "radius_se",
    "texture_se",
    "perimeter_se",
    "area_se",
    "smoothness_se",
    "compactness_se",
    "concavity_se",
    "concave_points_se",
    "symmetry_se",
    "fractal_dimension_se",

    "radius_worst",
    "texture_worst",
    "perimeter_worst",
    "area_worst",
    "smoothness_worst",
    "compactness_worst",
    "concavity_worst",
    "concave_points_worst",
    "symmetry_worst",
    "fractal_dimension_worst"
]

In [23]:
df = pd.read_csv(
    "/wdbc.data",
    header=None,
    names=columns
)

print("Dataset shape:", df.shape)

display(df.head())

Dataset shape: (569, 32)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [24]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDiagnosis distribution:")
print(df["diagnosis"].value_counts())

Dataset shape: (569, 32)

Missing values:
0

Diagnosis distribution:
diagnosis
B    357
M    212
Name: count, dtype: int64


In [25]:
df_model = df.drop(columns=["id"])

In [26]:
X = df_model.drop(columns=["diagnosis"])

y = df_model["diagnosis"].map({
    "B": 0,
    "M": 1
})

In [27]:
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget mapping:")
print("0 = Benign")
print("1 = Malignant")

print("\nTarget distribution:")
print(y.value_counts())

X shape: (569, 30)
y shape: (569,)

Target mapping:
0 = Benign
1 = Malignant

Target distribution:
diagnosis
0    357
1    212
Name: count, dtype: int64


In [28]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (341, 30)
Validation: (114, 30)
Test: (114, 30)


In [29]:
print("TRAIN")
print(y_train.value_counts())

print("\nVALIDATION")
print(y_val.value_counts())

print("\nTEST")
print(y_test.value_counts())

TRAIN
diagnosis
0    214
1    127
Name: count, dtype: int64

VALIDATION
diagnosis
0    71
1    43
Name: count, dtype: int64

TEST
diagnosis
0    72
1    42
Name: count, dtype: int64


In [30]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

Train: (341, 30)
Validation: (114, 30)
Test: (114, 30)


In [31]:
selector = SelectKBest(
    score_func=mutual_info_classif,
    k=8
)
X_train_selected = selector.fit_transform(
    X_train_scaled,
    y_train
)
X_val_selected = selector.transform(
    X_val_scaled
)
X_test_selected = selector.transform(
    X_test_scaled
)
print("After feature selection:")

print("Training:", X_train_selected.shape)

print("Validation:", X_val_selected.shape)

print("Test:", X_test_selected.shape)

After feature selection:
Training: (341, 8)
Validation: (114, 8)
Test: (114, 8)


In [32]:
selected_features = X.columns[selector.get_support()]

print("Selected 8 features:\n")

for i, feature in enumerate(selected_features, 1):
    print(f"{i}. {feature}")

Selected 8 features:

1. radius_mean
2. perimeter_mean
3. area_mean
4. concave_points_mean
5. radius_worst
6. perimeter_worst
7. area_worst
8. concave_points_worst


In [33]:
feature_scores = pd.DataFrame({
    "Feature": X.columns,
    "Mutual_Information_Score": selector.scores_
})

feature_scores = feature_scores.sort_values(
    by="Mutual_Information_Score",
    ascending=False
).reset_index(drop=True)

display(feature_scores)

,Feature,Mutual_Information_Score
0,concave_points_mean,0.458376
1,radius_worst,0.457444
2,area_worst,0.449047
3,perimeter_worst,0.447237
4,concave_points_worst,0.435871
5,perimeter_mean,0.414244
6,radius_mean,0.379015
7,area_mean,0.375809
8,concavity_mean,0.346917
9,area_se,0.346686


In [34]:
X_train_8 = pd.DataFrame(
    X_train_selected,
    columns=selected_features,
    index=X_train.index
)

X_val_8 = pd.DataFrame(
    X_val_selected,
    columns=selected_features,
    index=X_val.index
)

X_test_8 = pd.DataFrame(
    X_test_selected,
    columns=selected_features,
    index=X_test.index
)

In [35]:
train_final = X_train_8.copy()
train_final["target"] = y_train

val_final = X_val_8.copy()
val_final["target"] = y_val

test_final = X_test_8.copy()
test_final["target"] = y_test

In [36]:
train_final.to_csv(
    "WDBC_train_8_features.csv",
    index=False
)

val_final.to_csv(
    "WDBC_validation_8_features.csv",
    index=False
)

test_final.to_csv(
    "WDBC_test_8_features.csv",
    index=False
)

print("All processed datasets saved successfully!")

All processed datasets saved successfully!


In [39]:
feature_scores.to_csv(
    "WDBC_feature_ranking.csv",
    index=False
)

print("Saved: WDBC_feature_ranking.csv")
df.to_csv(
    "WDBC_raw.csv",
    index=False
)

print("Saved: WDBC_raw.csv")

Saved: WDBC_feature_ranking.csv
Saved: WDBC_raw.csv


In [40]:
from google.colab import files
files.download("WDBC_train_8_features.csv")
files.download("WDBC_validation_8_features.csv")
files.download("WDBC_test_8_features.csv")
files.download("WDBC_feature_ranking.csv")
files.download("WDBC_raw.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>